In [3]:
"""Tune DecisionTreeClassifier's min_samples_split on the mushroom dataset."""
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

BASE_DIR = Path(__file__).resolve().parent
DATA_PATH = BASE_DIR / "data/mushrooms.csv"
TARGET = "class"
RANDOM_STATE = 42


def main():
    df = pd.read_csv(DATA_PATH)

    if TARGET not in df.columns:
        raise ValueError(f"Target column '{TARGET}' was not found in the CSV file.")

    # Separate input features and target label.
    X = df.drop(columns=[TARGET])
    y = df[TARGET]

    # Keep the class proportions similar in train and test sets.
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    # All mushroom attributes are categorical, so encode them for sklearn.
    preprocessing = ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore"), X.columns.tolist())
        ]
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessing", preprocessing),
            ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE)),
        ]
    )

    # min_samples_split = minimum number of rows required to split a node.
    candidates = [2, 3, 5, 10, 15, 20, 30, 40, 50, 75, 100, 150, 200]
    parameter_grid = {"classifier__min_samples_split": candidates}

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=parameter_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        return_train_score=True,
        refit=True,
    )
    search.fit(X_train, y_train)

    # Evaluate the selected model once on the untouched test set.
    best_model = search.best_estimator_
    predictions = best_model.predict(X_test)

    print("Dataset shape:", df.shape)
    print("Class distribution:\n", y.value_counts())
    print("\nBest min_samples_split:", search.best_params_["classifier__min_samples_split"])
    print(f"Best 5-fold CV macro F1: {search.best_score_:.6f}")
    print(f"Test macro F1: {f1_score(y_test, predictions, average='macro'):.6f}")
    print(f"Test accuracy: {accuracy_score(y_test, predictions):.6f}")
    print("\nClassification report:\n", classification_report(y_test, predictions))

    # Save every tested value and its cross-validation scores.
    results = pd.DataFrame(
        {
            "min_samples_split": search.cv_results_["param_classifier__min_samples_split"].astype(int),
            "mean_cv_f1_macro": search.cv_results_["mean_test_score"],
            "std_cv_f1_macro": search.cv_results_["std_test_score"],
            "mean_train_f1_macro": search.cv_results_["mean_train_score"],
        }
    ).sort_values("min_samples_split")
    results.to_csv(BASE_DIR / "f1_results.csv", index=False)
    print(f"\nAll CV results saved to: {BASE_DIR / 'f1_results.csv'}")


if __name__ == "__main__":
    main()


NameError: name '__file__' is not defined